# Sprint 0 · ¿Qué clientes se van?

**Nivel Intermedio** · Sprint 0 · Se publica al final de la clase de la semana 5 · Se entrega al inicio de la clase de la semana 6 · Tiempo estimado: 3–4 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/tareas/Sprint0_abandono_clientes.ipynb)

Una compañía de teléfono e internet pierde cada mes a uno de cada cuatro clientes. Si supiera **quién se va a ir** antes de que se vaya, podría llamarlo y ofrecerle un descuento. Ese problema se llama **abandono de clientes** (*churn*) y es de los más comunes en la industria.

Tienes los datos de 5362 clientes con la respuesta (`Churn`: se fue o se quedó) y de otros 1761 clientes **sin** la respuesta. Tu trabajo: limpiar los datos, que vienen desordenados como en la vida real, entrenar modelos de sklearn, compararlos contra una línea base y predecir quién se va entre los 1761. El profesor tiene guardadas las respuestas de esos 1761 (el **conjunto de prueba oculto**) y en la clase de la semana 6 publica la **tabla de posiciones**.

**Objetivos.** Al terminar podrás:

1. Encontrar y arreglar los problemas típicos de un CSV real: texto mal escrito, números guardados como texto, valores imposibles, faltantes y filas repetidas.
2. Convertir columnas de texto en números con *one-hot encoding* sin filtrar información del conjunto de prueba.
3. Comparar al menos dos modelos con validación cruzada y reportar *accuracy*, precisión, *recall* y F1 contra una línea base.
4. Elegir un umbral de decisión con un conjunto de validación y entregar predicciones en el formato pedido.

**Prerrequisitos.** pandas (semana 3) y la clase de la semana 5 (la receta completa de su parte 7: línea base, validación cruzada, métricas). Las funciones de pandas nuevas se explican en la sección "Caja de herramientas".

**Cómo entregar.** Son **dos archivos** en tu repositorio `Alto-Rendimiento-IA/entregas-<tu-usuario>`, en la carpeta **`Sprint0/`**:

1. **El notebook.** Antes de escribir nada: *Archivo → Guardar una copia en GitHub*, rama `main`, ruta **`Sprint0/Sprint0_abandono_clientes.ipynb`**, mensaje `Empezar Sprint 0`. Trabaja en la copia que se abre y guarda ahí después de cada ejercicio.
2. **`predicciones.csv`**, el archivo que escribe este notebook. Colab no lo sube solo: al final te explicamos cómo subirlo desde la página de GitHub.

Lo que esté en tu repositorio al inicio de la clase de la semana 6 es tu entrega.

**Reglas del sprint.**

- **Es individual.** No compartas código ni puntajes con tus compañeros hasta que se publique la tabla.
- **Uso de IA**, como en los trabajos: puedes preguntar cómo se usa una función (`pd.to_numeric`, `pd.get_dummies`). No le pidas el código de la limpieza ni del modelo.
- **Solo estos datos.** Vienen de un ejemplo público de IBM (*Telco Customer Churn*, licencia Apache 2.0), modificado para este sprint: quitamos columnas, cambiamos los identificadores y lo desordenamos. Buscar las respuestas originales cuenta como copia.

## Preparación

Ejecuta esta celda primero. Carga las librerías, fija la semilla, descarga los dos CSV y define `save_predictions`, que escribe el archivo de entrega y revisa su formato.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 0
np.random.seed(SEED)

DATA_URL = "https://raw.githubusercontent.com/melvinpqbsc/Alto_Rendimiento_IA/main/Nivel_Intermedio/tareas/Sprint0_datos/"
train = pd.read_csv(DATA_URL + "train.csv")  # con la respuesta, Churn
test = pd.read_csv(DATA_URL + "test.csv")    # sin la respuesta: es lo que hay que predecir

TEXT_COLS = ["Partner", "Dependents", "PhoneService", "InternetService", "OnlineSecurity",
             "TechSupport", "StreamingTV", "Contract", "PaperlessBilling", "PaymentMethod"]
NUM_COLS = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]


def save_predictions(ids, pred, path="predicciones.csv"):
    """Escribe el archivo de entrega: una fila por cliente de test, columnas id y Churn (0 o 1)."""
    ids, pred = np.asarray(ids), np.asarray(pred).astype(int)
    assert len(ids) == len(test) and set(ids) == set(test["id"]), "hace falta una predicción por cada id de test"
    assert set(np.unique(pred)) <= {0, 1}, "las predicciones deben ser 0 (se queda) o 1 (se va)"
    pd.DataFrame({"id": ids, "Churn": pred}).to_csv(path, index=False)
    print(f"✓ {path}: {len(pred)} clientes, {pred.mean():.1%} marcados como 'se va'")


print("train:", train.shape, " test:", test.shape)
train.head()

## 1. Los datos y la métrica

Cada fila es un cliente. Los nombres de las columnas quedan en inglés, como en una competencia:

| Columna | Qué es |
|---|---|
| `id` | identificador del cliente |
| `SeniorCitizen` | 1 si el cliente es adulto mayor |
| `Partner`, `Dependents` | si tiene pareja; si tiene personas a su cargo |
| `tenure` | meses que lleva como cliente |
| `PhoneService` | si tiene línea de teléfono |
| `InternetService` | tipo de internet: DSL, fibra óptica (*fiber optic*) o ninguno |
| `OnlineSecurity`, `TechSupport`, `StreamingTV` | servicios extra contratados (o "sin internet") |
| `Contract` | contrato mes a mes, de un año o de dos años |
| `PaperlessBilling` | si recibe la factura por correo electrónico |
| `PaymentMethod` | forma de pago |
| `MonthlyCharges` | lo que paga cada mes, en dólares |
| `TotalCharges` | lo que pagó en total hasta hoy, en dólares |
| `Churn` | **la respuesta**: `Yes` si se fue, `No` si se quedó (solo en `train`) |

**La métrica de la tabla de posiciones es F1 de la clase "se va"** (semana 5). Solo uno de cada cuatro clientes se va, así que un modelo que dice "todos se quedan" acierta el 74% sin servir para nada: su F1 es 0. Repaso, con VP = verdaderos positivos (dijiste "se va" y se fue), FP = falsos positivos y FN = falsos negativos:

$$\text{precisión} = \frac{VP}{VP + FP} \qquad \text{recall} = \frac{VP}{VP + FN} \qquad F_1 = \frac{2 \cdot \text{precisión} \cdot \text{recall}}{\text{precisión} + \text{recall}}$$

## 2. La línea base (ya funciona)

Antes de limpiar nada, un modelo simple: un árbol de profundidad 3 que solo mira `tenure` y `MonthlyCharges`, con los faltantes rellenados por la mediana. Los árboles no necesitan escalar, por eso aguanta los datos sucios. Lo medimos con **validación cruzada de 5 pliegues** (semana 5): es tu **validación local**, la que reemplaza a la prueba oculta mientras trabajas.

Esta celda también escribe un `predicciones.csv` con la línea base. Si lo subes tal cual, ya estás en la tabla. Tu objetivo es superarla.

In [ ]:
BASE_COLS = ["tenure", "MonthlyCharges"]
y_raw = (train["Churn"] == "Yes").astype(int)
fill = train[BASE_COLS].median()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
baseline = DecisionTreeClassifier(max_depth=3, random_state=SEED)

for name, model in [("siempre se queda", DummyClassifier(strategy="most_frequent")),
                    ("línea base: árbol, 2 columnas", baseline)]:
    acc = cross_val_score(model, train[BASE_COLS].fillna(fill), y_raw, cv=cv, scoring="accuracy")
    f1 = cross_val_score(model, train[BASE_COLS].fillna(fill), y_raw, cv=cv, scoring="f1")
    print(f"{name:<32} accuracy {acc.mean():.3f}   F1 {f1.mean():.3f} ± {f1.std():.3f}")

baseline.fit(train[BASE_COLS].fillna(fill), y_raw)
save_predictions(test["id"], baseline.predict(test[BASE_COLS].fillna(fill)))

## 3. Caja de herramientas de limpieza

Funciones de pandas que no vimos en la semana 3, probadas en una tabla de juguete con los mismos problemas que vas a encontrar. Ejecuta y lee cada línea.

In [ ]:
toy = pd.DataFrame({
    "plan": [" Basic", "basic", "PREMIUM ", "Premium", " Basic"],
    "senior": ["0", "Yes", "1", "No", "0"],
    "price": ["$1,200.50", "80", " ", "95.5", "$1,200.50"],
    "months": [12, -1, 5, 40, 12],
})
print(toy, "\n")

# texto: quitar espacios de los bordes y pasar a minúsculas (los NaN se quedan como NaN)
print(toy["plan"].str.strip().str.lower().tolist())

# cambiar unos valores por otros
print(toy["senior"].replace({"No": "0", "Yes": "1"}).astype(int).tolist())

# texto → número: primero quitar "$" y ",", después convertir; lo que no se puede convertir queda NaN
clean_price = toy["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
print(pd.to_numeric(clean_price, errors="coerce").tolist())

# un valor imposible → faltante, con una máscara (semana 3)
months = toy["months"].astype(float)
months[months < 0] = np.nan
print(months.tolist())

# filas repetidas
print("repetidas:", toy.duplicated().sum(), "→ después de drop_duplicates:", len(toy.drop_duplicates()), "filas")

Los modelos solo entienden números. Una columna de texto como `plan` se convierte con *one-hot encoding* (codificación "uno caliente"): una columna de 0 y 1 por cada valor posible. ¿Por qué no numerar los valores (basic = 0, premium = 1, gold = 2)? Porque eso inventa un orden y una distancia que no existen: con `PaymentMethod`, "cheque" no está "entre" "tarjeta" y "transferencia".

In [ ]:
plans_train = pd.DataFrame({"plan": ["basic", "premium", "gold", "basic"]})
plans_test = pd.DataFrame({"plan": ["basic", "basic"]})  # en test no aparece ningún premium ni gold

d_train = pd.get_dummies(plans_train, dtype=float)
d_test = pd.get_dummies(plans_test, dtype=float)
print(d_train, "\n")
print("columnas de test antes:", d_test.columns.tolist())

# test debe tener exactamente las mismas columnas que train, en el mismo orden: reindex rellena con 0
d_test = d_test.reindex(columns=d_train.columns, fill_value=0)
print(d_test)

## Ejercicio 1: el diagnóstico

Antes de arreglar, hay que encontrar los problemas. Explora `train` con lo de la semana 3 y la caja de herramientas:

- `train.dtypes`: ¿qué columnas numéricas quedaron como texto (`object`)?
- `train[col].value_counts(dropna=False)` para cada columna de `TEXT_COLS` y para `SeniorCitizen`: ¿el mismo valor escrito de varias formas?
- `train.isna().sum()` y `train.duplicated().sum()`.
- `train[["tenure", "MonthlyCharges"]].describe()` y un histograma de `MonthlyCharges`: ¿valores imposibles? Pista: un plan de teléfono no cuesta 7000 dólares al mes.
- Mira a mano algunos valores de `TotalCharges`.

Hay **al menos seis** problemas distintos.

In [ ]:
# TODO: explora train

**Los problemas que encontré** (columna, qué pasa, cuántas filas, cómo lo voy a arreglar):

1. *Escribe aquí tu lista.*

**Guarda en GitHub** con el mensaje `Ejercicio 1 resuelto`.

## Ejercicio 2: la función de limpieza

Escribe **una sola** función `clean(df)` que sirva para `train` y para `test`: así los dos quedan limpios de la misma forma. Devuelve una tabla nueva (empieza con `df = df.copy()`) y **no borra filas**, porque en `test` hay que predecir todas. Pasos:

1. En cada columna de `TEXT_COLS`: quitar espacios de los bordes y pasar a minúsculas.
2. `SeniorCitizen`: entero, 0 o 1.
3. `tenure`: los valores negativos pasan a NaN.
4. `MonthlyCharges`: los valores mayores que 200 están en centavos; divídelos por 100.
5. `TotalCharges`: número (float). Quita `$` y `,` antes de convertir; los vacíos quedan NaN.
6. `Churn`, solo si la columna existe (`if "Churn" in df.columns`): entero, 1 si se fue y 0 si se quedó.

Después, fuera de la función, quita las filas repetidas de `train_clean` (paso 7). ¿Por qué no se quitan dentro de `clean`?

In [ ]:
def clean(df):
    df = df.copy()
    # TODO: pasos 1 a 6
    return df


train_clean = clean(train)  # TODO: además, quita las filas repetidas
test_clean = clean(test)
print("train_clean:", train_clean.shape, " test_clean:", test_clean.shape)

In [ ]:
# Verificación
assert len(test_clean) == len(test) and (test_clean["id"].to_numpy() == test["id"].to_numpy()).all(), \
    "test_clean debe tener las mismas filas que test, en el mismo orden: nunca borres filas de test"
assert train_clean.duplicated().sum() == 0, "todavía hay filas repetidas en train_clean"
assert len(train_clean) == 5282, f"train_clean debería tener 5282 filas, tiene {len(train_clean)}"
for name, d in [("train_clean", train_clean), ("test_clean", test_clean)]:
    for col in NUM_COLS:
        assert pd.api.types.is_numeric_dtype(d[col]), f"{name}: {col} todavía no es numérica"
    for col in TEXT_COLS:
        values = d[col].dropna()
        assert (values == values.str.strip().str.lower()).all(), f"{name}: {col} tiene mayúsculas o espacios"
    assert set(d["SeniorCitizen"]) == {0, 1}, f"{name}: SeniorCitizen debe tener solo 0 y 1"
    assert d["tenure"].min() >= 0, f"{name}: tenure todavía tiene valores negativos"
    assert d["MonthlyCharges"].max() < 200, f"{name}: MonthlyCharges todavía tiene valores en centavos"
    assert d["TotalCharges"].isna().mean() < 0.01, \
        f"{name}: TotalCharges tiene demasiados faltantes: ¿quitaste $ y , antes de convertir?"
assert set(d["Contract"].dropna()) == {"month-to-month", "one year", "two year"}
assert set(train_clean["Churn"]) == {0, 1}, "Churn debe tener solo 0 y 1"
print(f"✓ limpieza correcta. Se van {train_clean['Churn'].mean():.1%} de los clientes de train.")

**Guarda en GitHub** con el mensaje `Ejercicio 2 resuelto`.

## Ejercicio 3: faltantes y *one-hot encoding*

Completa `make_features(df, medians)`, que convierte una tabla limpia en la matriz de números que recibe sklearn:

1. Quita `id` (no dice nada del cliente) y `Churn` (es la respuesta). Ya está hecho.
2. Rellena los faltantes de `NUM_COLS` con `medians`.
3. Rellena los faltantes de `TEXT_COLS` con el texto `"desconocido"`: no saber la forma de pago también es información.
4. *One-hot*: `pd.get_dummies(X, columns=TEXT_COLS, dtype=float)`.

Las medianas se calculan **solo con train**, y la misma se usa para test: es la regla de la fuga de datos de la semana 5. La prueba representa clientes del futuro, y el futuro no se puede mirar para preparar los datos.

In [ ]:
def make_features(df, medians):
    X = df.drop(columns=["id", "Churn"], errors="ignore")
    # TODO: pasos 2, 3 y 4
    return X


medians = train_clean[NUM_COLS].median()
X = make_features(train_clean, medians)
X_test = make_features(test_clean, medians).reindex(columns=X.columns, fill_value=0)
y = train_clean["Churn"].to_numpy()
print("X:", X.shape, " X_test:", X_test.shape, " y:", y.shape)

In [ ]:
# Verificación
for name, M in [("X", X), ("X_test", X_test)]:
    assert M.isna().sum().sum() == 0, f"{name} todavía tiene faltantes"
    assert all(pd.api.types.is_numeric_dtype(t) for t in M.dtypes), f"{name} todavía tiene columnas de texto"
    assert "id" not in M.columns and "Churn" not in M.columns, f"{name} no debe tener id ni Churn"
assert "PaymentMethod_desconocido" in X.columns, "falta la categoría 'desconocido' en PaymentMethod"
assert X.shape == (5282, 32), f"X debería ser (5282, 32), es {X.shape}"
assert X_test.shape == (1761, 32) and list(X_test.columns) == list(X.columns)
print("✓ X y X_test listos:", X.shape[1], "columnas, todas numéricas y sin faltantes")

**Guarda en GitHub** con el mensaje `Ejercicio 3 resuelto`.

## Ejercicio 4: dos modelos contra la línea base

Agrega a `candidates` **al menos dos modelos distintos** de la semana 5 (regresión logística, kNN, árbol de decisión), **cada uno con al menos dos valores de su perilla principal** (`C`, `n_neighbors`, `max_depth`). Recuerda: la regresión logística y kNN van en un *pipeline* con `StandardScaler`; el árbol no lo necesita.

Después completa el bucle: para cada candidato, F1 con validación cruzada (`cross_val_score` con el mismo `cv` de la línea base y `scoring="f1"`). Guarda el promedio en `cv_f1[name]` e imprime promedio ± desviación estándar.

In [ ]:
candidates = {
    "siempre se queda": DummyClassifier(strategy="most_frequent"),
    # TODO: al menos dos modelos, cada uno con al menos dos valores de su perilla
}

cv_f1 = {}
for name, model in candidates.items():
    # TODO: F1 con validación cruzada; guarda el promedio en cv_f1[name] e imprímelo con la desviación
    pass

La línea base, medida ahora sobre los datos limpios (las mismas filas que tus modelos, así la comparación es justa):

In [ ]:
base_cv = cross_val_score(baseline, X[BASE_COLS], y, cv=cv, scoring="f1").mean()
print(f"línea base (árbol, 2 columnas), datos limpios: F1 {base_cv:.3f}")

# Verificación
models_tried = [name for name in cv_f1 if name != "siempre se queda"]
assert len(models_tried) >= 4, "hacen falta al menos dos modelos con dos valores de perilla cada uno"
assert all(0 <= v <= 1 for v in cv_f1.values()), "cv_f1 debe guardar promedios de F1, entre 0 y 1"
best_cv = max(cv_f1[name] for name in models_tried)
assert best_cv > base_cv, f"tu mejor modelo ({best_cv:.3f}) todavía no supera a la línea base ({base_cv:.3f})"
print(f"✓ tu mejor modelo supera a la línea base por {best_cv - base_cv:.3f} de F1")

**Guarda en GitHub** con el mensaje `Ejercicio 4 resuelto`.

## Ejercicio 5: el informe de validación y el umbral

La validación cruzada eligió un modelo. Ahora lo miramos de cerca con un **conjunto de validación**: el 20% de train, separado y estratificado. Es tu espejo de la prueba oculta.

**a)** Entrena una copia nueva del mejor modelo con `X_dev, y_dev` (`clone(candidates[best_name]).fit(...)`) y calcula `proba_val`, la probabilidad de "se va" para cada cliente de `X_val` (`predict_proba(...)[:, 1]`).

**b)** Arma `val_table`: un `DataFrame` con una fila por modelo y las columnas `accuracy`, `precision`, `recall` y `f1`, todas en validación. Filas: "siempre se queda", la línea base (árbol con `BASE_COLS`, entrenado con `X_dev[BASE_COLS]`) y tu mejor modelo con el umbral 0.5 (`proba_val >= 0.5`). La función `scores` ya calcula las cuatro métricas.

In [ ]:
X_dev, X_val, y_dev, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
best_name = max(cv_f1, key=cv_f1.get)
print("mejor modelo en validación cruzada:", best_name)


def scores(y_true, pred):
    return {"accuracy": accuracy_score(y_true, pred), "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred), "f1": f1_score(y_true, pred)}


best = ...        # TODO a
proba_val = ...   # TODO a
val_table = ...   # TODO b
val_table

**c)** El modelo dice "se va" cuando la probabilidad pasa de 0.5, pero ese umbral no tiene nada de especial (semana 5). Para cada umbral de `thresholds`, calcula F1, precisión y *recall* en validación y dibuja las tres curvas contra el umbral, con ejes y leyenda. Guarda en `best_threshold` el umbral con mejor F1 y agrega a `val_table` una fila con tu modelo y ese umbral.

Elegimos el umbral con validación, **nunca** con la prueba: la prueba se mira una sola vez, al final.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
# TODO c: F1, precisión y recall para cada umbral, y el gráfico
best_threshold = ...  # TODO c

In [ ]:
# Verificación
assert isinstance(best_threshold, float) and 0 < best_threshold < 1, "best_threshold debe ser un número entre 0 y 1"
assert len(proba_val) == len(y_val) and ((proba_val >= 0) & (proba_val <= 1)).all(), "proba_val: una probabilidad por cliente de X_val"
assert isinstance(val_table, pd.DataFrame) and len(val_table) >= 4, "val_table: al menos 4 filas"
assert {"accuracy", "precision", "recall", "f1"} <= set(val_table.columns)
f1_best, f1_half = f1_score(y_val, proba_val >= best_threshold), f1_score(y_val, proba_val >= 0.5)
assert f1_best >= f1_half - 1e-9, "con el mejor umbral el F1 no puede ser peor que con 0.5"
print(f"✓ umbral {best_threshold:.2f}: F1 en validación {f1_best:.3f} (con 0.5 era {f1_half:.3f})")

**Guarda en GitHub** con el mensaje `Ejercicio 5 resuelto`.

## Ejercicio 6: las predicciones finales

El paso 4 de la receta: reentrena el modelo elegido con **todo** train (`X, y`), no solo con `X_dev`, porque más datos casi siempre ayudan. Calcula la probabilidad de "se va" para `X_test`, aplica `best_threshold` y guarda el resultado con `save_predictions(test_clean["id"], pred_test)`.

In [ ]:
final = ...      # TODO
pred_test = ...  # TODO: 0 o 1 para cada cliente de test
# TODO: save_predictions

In [ ]:
# Verificación
sub = pd.read_csv("predicciones.csv")
assert list(sub.columns) == ["id", "Churn"], "las columnas deben ser id y Churn"
assert len(sub) == len(test) and set(sub["id"]) == set(test["id"]), "una fila por cada id de test"
assert set(sub["Churn"]) <= {0, 1}
assert (sub["Churn"].to_numpy() == np.asarray(pred_test)).all(), \
    "predicciones.csv no tiene tus predicciones finales: ¿llamaste a save_predictions?"
print(f"✓ predicciones.csv listo: {len(sub)} clientes, {sub['Churn'].mean():.1%} marcados como 'se va' "
      f"(en train se fue el {y.mean():.1%})")

### Subir `predicciones.csv` a tu repositorio

1. Ejecuta la celda de abajo: en Colab, descarga el archivo a tu computadora.
2. Abre `https://github.com/Alto-Rendimiento-IA/entregas-<tu-usuario>` y entra a la carpeta `Sprint0`.
3. *Add file → Upload files*, arrastra `predicciones.csv` y pulsa *Commit changes*.

Si después mejoras tu modelo, repite los tres pasos: el archivo nuevo reemplaza al anterior. Vale el que esté en el repositorio al inicio de la clase de la semana 6.

In [ ]:
try:
    from google.colab import files
    files.download("predicciones.csv")
except ImportError:
    print("No estás en Colab: el archivo está en", Path("predicciones.csv").resolve())

**Guarda en GitHub** con el mensaje `Ejercicio 6 resuelto`.

## Ejercicio 7: preguntas

Responde con tus números y tus gráficos, en dos a cuatro oraciones cada una.

**a)** El modelo "siempre se queda" tiene un *accuracy* cercano al 74%. ¿Por qué no sirve, y qué número de `val_table` lo delata?

**b)** ¿Qué paso de la limpieza cambió más el resultado? Pruébalo: quita un paso de `clean` (por ejemplo, no pases el texto a minúsculas), vuelve a ejecutar los ejercicios 2 a 4 y compara el F1 de validación cruzada de tu mejor modelo. Deja la función como estaba al terminar.

**c)** ¿Qué umbral elegiste y qué pasó con la precisión y el *recall* respecto del umbral 0.5? Para la compañía, ¿qué cuesta un falso positivo (llamar con un descuento a alguien que no se iba) y qué cuesta un falso negativo (no llamar a alguien que se va)?

**d)** Tu F1 en validación cruzada y tu F1 en la validación del ejercicio 5 no son iguales. ¿En cuál confías más para adivinar tu puntaje en la tabla de posiciones, y por qué?

*Escribe aquí tus respuestas.*

**Guarda en GitHub** con el mensaje `Ejercicio 7 resuelto`.

## Conclusión

*Escribe aquí un párrafo en lenguaje sencillo: qué modelo elegiste y por qué, qué significan tus números comparados con la línea base, y qué probarías después para subir en la tabla. Sin este párrafo el trabajo no tiene nota.*

## Uso de IA

*Una línea: para qué usaste IA en este sprint, o "No usé IA".*

## Rúbrica

Asígnate tu puntaje en la columna "Yo". El profesor completa la otra columna. La tabla de posiciones (F1 en la prueba oculta) se publica en la clase de la semana 6 y no cambia esta nota: superar a la línea base y explicar bien vale más que el primer puesto.

| Criterio | Qué se mira | Yo (0–2) | Profesor (0–2) |
|---|---|---|---|
| **Se ejecuta** | *Entorno de ejecución → Reiniciar y ejecutar todo* corre sin errores | | |
| **Es correcto** | Todas las verificaciones imprimen ✓ y `Sprint0/predicciones.csv` está en tu repositorio con el formato pedido | | |
| **Está justificado** | La lista del ejercicio 1, el gráfico del ejercicio 5 y las respuestas del ejercicio 7 usan tus números | | |
| **Está bien comunicado** | La conclusión se entiende sin leer el código; hay un *commit* por ejercicio con un mensaje claro | | |
| **Total** | | /8 | /8 |

**Antes de la clase de la semana 6**: ejecuta *Entorno de ejecución → Reiniciar y ejecutar todo*, revisa que todo imprima ✓, guarda por última vez en `Sprint0/Sprint0_abandono_clientes.ipynb` y sube el `predicciones.csv` de esa última ejecución.